# COMP551 PyTorch Tutorial

**Instructor:** He Zhu, Mike

**Date:** 29 Oct, 2026 11:30 am -13:00 pm

**Room:** MC 103

## Tutorial Outline

* Warm-up
* Load Dataset
* Define PyTorch Model
* Train Model
* Test Model


## Warm-up

First, we need to make sure that GPU is available. Training models with GPU is *much* faster than CPU.

In Google Colab, you can select **Runtime -> Change runtime type** to select a GPU environment.

### **Exercise 1:** Try switching between CPU, GPU, and TPU in Colab. Then compare the outputs below.

In [ ]:
import torch

# True: using GPU. False: using CPU.
torch.cuda.is_available()

True

We need to choose a computation device (CPU or GPU), and move both the model and the data to it. During training, all computations will run on that device. We can simply set it by checking whether GPU is available or not in our environment.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# later: model = model.to(device)
# later: x, y = x.to(device), y.to(device)

## Load Dataset

The easiest way to load a dataset is to use PyTorch’s built-in datasets.

**Computer Vision:**

`TorchVision` provides built-in datasets for computer vision tasks. The most popular datasets include:
* CIFAR10
* MNIST
* FashionMNIST

Reference:
https://docs.pytorch.org/vision/main/datasets.html

**Natual Language Processing:**

`TorchText` provides built-in datasets for natual language processing tasks. The most popular datasets include:
* IMDB

Reference:
https://docs.pytorch.org/text/master/datasets.html


### **Step 1: Load Dataset**

In [ ]:
from torchvision import datasets
from torchvision import transforms

# Load an MNIST dataset using TorchVision. You can also try CIFAR10, FashionMNIST, etc.
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

A PyTorch tensor is a multidimensional array. It can be a scalar, a vector, a matrix, or a higher-dimensional array.
It supports GPU acceleration and automatic differentiation (Autograd).

![PyTorch Tensor Examples](https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/tensor_examples.png)

**(1) Commonly used properties for tensor:**

- `shape`: shape of the tensor

- `dtype`: data type of the tensor

- `device`: device where the tensor is stored (CPU / GPU)

In [ ]:
image, label = train_data[0]
print(type(image))
print(image.shape)
print(image.dtype)
print(image.device)

<class 'torch.Tensor'>
torch.Size([1, 28, 28])
torch.float32
cpu


**(2) Commonly used methods for tensor:**
- `to(device)`: move the tensor to a different device (CPU / GPU)

- `backward()`: backpropagation and compute gradients

In [ ]:
image = image.to(device)
print(image.device)

cuda:0


### **Step 2: Normalize Dataset**

In [ ]:
from torch.utils.data import DataLoader

sum = 0.0
squared_sum = 0.0
N = 0
train_loader = DataLoader(train_data, batch_size=1000, shuffle=False)
for data, _ in train_loader: # Accumulative sum and std throughout data
    sum += data.sum(dim=[0, 2, 3])
    squared_sum += (data ** 2).sum(dim=[0, 2, 3])
    N += data.numel()
mean = (sum / N).item()
std = (torch.sqrt((squared_sum / N) - (mean ** 2))).item()
print(mean)
print(std)

0.13066045939922333
0.30810782313346863


Let's take a closer look at this code:

```
sum += data.sum(dim=[0, 2, 3])
```

Our data is a four-dimensional tensor: (batch_size, channel, height, width).

So `dim=[0, 2, 3]` means we're summing over the 0-th (batch size), 2-nd (height), and 3-rd (width) dimensions.

Notice that we're not summing over 1-st dimension (channel dimension), because we want to keep the channel dimension.

For grayscale images, we only have one channel, so the result contains just one value.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])

In [ ]:
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

### **Step 3: Create Training and Validation Set**


To choose the best hyper-parameters during training and early stop the training process, we need to split the training data into training and validation sets.

In [ ]:
from torch.utils.data import random_split

split_ratio = 0.8
train_size = int(split_ratio * len(train_data))
val_size = len(train_data) - train_size

train_dataset, val_dataset = random_split(train_data, [train_size, val_size])

print(f"Train size: {train_size}")
print(f"Validation size: {val_size}")

Train size: 48000
Validation size: 12000


### **Step 4: Create Data Loaders**

Data loader is used to iterate mini-batches in each iteration, so that we can apply mini-batch training.
The constructor contains three arguments:
* `dataset`: The dataset that we aim to provide mini-batches.
* `batch_size`: The batch size of each mini-batch.
* `shuffle`:
    * `True`: Shuffle the data at the start of each iteration. This is typically used for the training set to add randomness and prevent the model from being affected by the order of the training data.
    * `False`: Keep a fixed order. This is typically used for validation/testing and we are only evaluating the model's performance.



In [ ]:
from torch.utils.data import DataLoader

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

### **Exercise 2:**

Refer to `train_data` and `train_loader`. Load MNIST test dataset and create a loader with batch size 64.

In [ ]:
# TODO: create test_data, then create test_loader
test_data = None
test_loader = None

assert test_data is not None, "Please implement test_data"
assert test_loader is not None, "Please implement test_loader"

## Define PyTorch Model

* First, we need to define a class (e.g. `MnistNetwork`) for our model. The class should inherit from the parent class `nn.Module`.
* Implement two key public methods:
    *   `def __init__(self)`: In the constructor, we need to define the model layers (e.g. fully-connected layers, dropout layers, etc.) or any parameters.
    *   `def forward(self, x)`: The `forward` function defines the forward pass: how the input `x` flows through layers and creates the outputs.



<img src="https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/model_class.png" width="900">

### **Exercise 3:**
* Currently, this is a 3-layer fully-connected neural network in the `forward` function.
* Implement the 4-th layer in the constructor and the `forward` function (`fc1` -> `fc2` -> `fc3` -> `fc4`).

In [ ]:
import torch.nn as nn

class MnistNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MnistNetwork, self).__init__()
        self.input_dim = input_dim

        # First layer: input dim: input_dim, output dim: hidden_dim
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # Second layer: input dim: hidden_dim, output dim: hidden_dim
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: Question: What is the input dim and output dim of the third layer?
        self.fc3 = nn.Linear(hidden_dim, output_dim)

        # TODO: define the 4-th layer here. Be aware of the output dimension of fc3 and input dimension of fc4.
        self.fc4 = None
        assert self.fc4 is not None, "Please implement self.fc4"

        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Reshape the image from [B, 1, 28, 28] to [B, 784]
        # x.view(-1, self.input_dim)  # -1 means infer the 0-th dimension automatically
        # e.g. x.view(B, -1, D)    # Output 3D tensor by inferring the middle dimension
        x = x.view(-1, self.input_dim)
        x = self.fc1(x)
        x = torch.relu(x)

        x = self.dropout(x)
        x = self.fc2(x)
        x = torch.relu(x)

        x = self.dropout(x)
        x = self.fc3(x)

        # TODO: implement the 4-th layer forward pass
        x = None
        assert x is not None, "Please implement x"

        return x

## Train Model

### **Step 1: Define the Hyper-parameters**

First, we need to define hyper-parameters such as learning rate, momentum, dropout rate, hidden dimensions, etc.

In [ ]:
# Define the hyper-parameters
lr = 0.01
momentum = 0.9
weight_decay = 1e-5
hidden_dim = 1200
data_shape = (1, 28, 28)
num_classes = 10

### **Step 2: Initialize the Model**

Second, we need to initialize the model and move it to the computation device (GPU / CPU).

In [ ]:
# Initialize the model
model = MnistNetwork(
    input_dim=data_shape[0] * data_shape[1] * data_shape[2],
    hidden_dim=hidden_dim,
    output_dim=num_classes
)

# Move the model to computation device (GPU / CPU)
model = model.to(device)

### **Step 3: Define the Optimizer**

Third, we need to define the optimizer with learning rate, weight decay, etc. We also need to specify the model parameters that we want to optimize.

Commonly used optimizers:
*   `optim.SGD`
*   `optim.Adam`
*   `optim.AdamW`



In [ ]:
import torch.optim as optim

# Define the optimizer
optimizer = optim.SGD(
    model.parameters(), # Important: Specify the model parameters
    lr=lr,
    momentum=momentum,
    weight_decay=weight_decay
)

### **Step 4 Specify the Loss Function:**

<img src="https://raw.githubusercontent.com/mikemikezhu/COMP551_McGill/main/assets/loss_function.png" width="900">

In [ ]:
# MNIST is an image classification task
criterion = nn.CrossEntropyLoss()

### **Step 5: Train Loop**

In [ ]:
epochs = 5

train_loss_history = []
val_loss_history = []
for epoch in range(epochs):

    """ Train on training set """

    # Set to training mode
    # When setting the model to training mode using model.train(),
    # some modules will take effect (e.g. dropout),
    # whereas setting the model to evaluation mode model.eval(),
    # then we will deactivate some modules (e.g. dropout).
    # Ref: https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval
    model.train()
    train_loss_accum = 0.0

    # Loop each mini-batch in train loader
    for batch_idx, (data, target) in enumerate(train_loader):

        # 1. Move the data to computation device (GPU / CPU)
        data, target = data.to(device), target.to(device)

        # 2. Making all gradients zero
        optimizer.zero_grad()

        # 3. Forward pass
        output = model(data)

        # 4. Calculate loss
        loss = criterion(output, target)

        # 5. Backward propogation and calculate gradients
        loss.backward()

        # 6. Update model parameters
        optimizer.step()

        train_loss_accum += loss.item()

    train_loss = train_loss_accum / len(train_loader)
    train_loss_history.append(train_loss)

    """ Evaluate on validation set """
    # Set to evaluation mode
    model.eval()

    val_loss_accum = 0.0

    # Prevent PyTorch from storing the computation graph and computing gradients -> save memory and computation
    with torch.no_grad():

        # Loop each mini-batch in val loader
        for data, target in val_loader:

            # 1. Move the data to computation device (GPU / CPU)
            data, target = data.to(device), target.to(device)

            # 2. Forward pass
            output = model(data)

            # 3. Calculate loss
            loss = criterion(output, target)

            val_loss_accum += loss.item()

    val_loss = val_loss_accum / len(val_loader)
    val_loss_history.append(val_loss)

    print(f'Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

### **Step 6: Plot Training Curve**

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(train_loss_history) + 1)

plt.figure(figsize=(6, 4))
plt.plot(epochs_range, train_loss_history, label="Train Loss")
plt.plot(epochs_range, val_loss_history, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Test Model

### **Exercise 4:**

After training, evaluate your model on the test set to measure its final performance. The testing procedure is almost identical to validation.

In [ ]:
model.eval()

all_preds = []
all_targets = []

correct = 0
total = 0

test_loss_accum = 0.0

# Prevent PyTorch from storing the computation graph and computing gradients -> save memory and computation
with torch.no_grad():

    # Loop each mini-batch in test loader
    for data, target in test_loader:

        # TODO: 1. Move the data to computation device (GPU / CPU)
        data, target = None, None
        assert (data is not None) and (target is not None), "Please implement data and target"

        # TODO: 2. Forward pass
        output = None
        assert output is not None, "Please implement output"

        # TODO: 3. Calculate loss
        loss = None
        assert loss is not None, "Please implement loss"

        test_loss_accum += loss.item()

        # 4. Calculate accuracy
        _, predicted = output.max(1)
        correct += (predicted == target).sum().item() # Number of correct predictions
        total += target.size(0)

test_loss = test_loss_accum / len(test_loader)

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.4f}%, Test Loss: {test_loss:.4f}')

### **Exercise 5:**
Play with the hyper-parameters. Check how the test performance changes with different learning rate, momentum, weight decay, training epochs, etc.

## Key Takeaway

- Preparation: Switch to GPU environment (if available) to accelerate training
- Load dataset:
    - PyTorch's built-in datasets (e.g. `TorchVision` for computer vision tasks and `TorchText` for natual language processing tasks)
    - PyTorch tensor: multidimensional array that supports GPU acceleration and AutoGrad
    - Normalize dataset
    - Create training and validation set
    - Create data loaders (e.g. `DataLoader`) to apply mini-batch training
- Define PyTorch model
    - Inherit PyTorch built-in class `nn.Module`
    - Implement the constructor and `forward()` function
- Train model
    - Define the hyper-parameters (e.g. learning rate, etc.)
    - Initialize the model
    - Define the optimizer (e.g. `optim.SGD(model.parameters(), lr=...)`)
    - Specify the loss function (e.g. `nn.CrossEntropyLoss()`)
    - Implement the training loop (e.g. `loss.backward()` $→$ `optimizer.step()`)
    - Plot the training curve
- Test model